# Support Vector Machine (SVM) from Scratch ⚔️

In this notebook, we implement a **Linear Support Vector Machine (SVM)** classifier using Gradient Descent.

## 📖 Theoretical Background

SVM aims to find a hyperplane that separates classes with the maximum margin.

### 1. The Hypothesis and Margin
For binary classification, labels $y$ must be encoded as $\{-1, 1\}$. The prediction is based on the sign of the linear model:
$$\hat{y} = \text{sign}(Xw - b)$$

### 2. Hinge Loss
The objective is to maximize the margin while penalizing points that fall on the wrong side of the margin. We use **Hinge Loss**:
$$J = \lambda ||w||^2 + \frac{1}{n} \sum_{i=1}^{n} \max(0, 1 - y_i(w \cdot x_i - b))$$

### 3. Gradients
If $y_i(w \cdot x_i - b) \geq 1$, the point is correctly classified and outside the margin. The gradient is just the regularization term:
$$dw = 2\lambda w, \quad db = 0$$

If $y_i(w \cdot x_i - b) < 1$, the point incurs a loss. The gradients are:
$$dw = 2\lambda w - y_ix_i$$
$$db = y_i$$

We update weights iteratively using these gradients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 🏗️ The Implementation

In [ ]:
class SVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iterations = n_iterations
        self.w = None
        self.b = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Encode labels as -1 and 1
        y_ = np.where(y <= 0, -1, 1)
        
        self.w = np.zeros(n_features)
        self.b = 0
        
        for _ in range(self.n_iterations):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]
                    
    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)


## 🧪 Data Generation and Training

In [ ]:
X, y = make_blobs(n_samples=250, centers=2, random_state=42, cluster_std=1.2)
y = np.where(y == 0, -1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

clf = SVM(learning_rate=0.001, lambda_param=0.01, n_iterations=1000)
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

accuracy = np.sum(predictions == y_test) / len(y_test)
print(f"SVM Accuracy: {accuracy * 100:.2f}%")

## 📊 Visualization

In [ ]:
def visualize_svm():
    def get_hyperplane_value(x, w, b, offset):
        return (-w[0] * x + b + offset) / w[1]

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(1,1,1)
    plt.scatter(X[:,0], X[:,1], marker='o', c=y, cmap="coolwarm", edgecolors="k")

    x0_1 = np.amin(X[:,0])
    x0_2 = np.amax(X[:,0])

    x1_1 = get_hyperplane_value(x0_1, clf.w, clf.b, 0)
    x1_2 = get_hyperplane_value(x0_2, clf.w, clf.b, 0)
    x1_1_m = get_hyperplane_value(x0_1, clf.w, clf.b, -1)
    x1_2_m = get_hyperplane_value(x0_2, clf.w, clf.b, -1)
    x1_1_p = get_hyperplane_value(x0_1, clf.w, clf.b, 1)
    x1_2_p = get_hyperplane_value(x0_2, clf.w, clf.b, 1)

    ax.plot([x0_1, x0_2], [x1_1, x1_2], 'y--')
    ax.plot([x0_1, x0_2], [x1_1_m, x1_2_m], 'k')
    ax.plot([x0_1, x0_2], [x1_1_p, x1_2_p], 'k')

    x1_min = np.amin(X[:,1])
    x1_max = np.amax(X[:,1])
    ax.set_ylim([x1_min - 3, x1_max + 3])
    plt.title("SVM Decision Boundary and Margins")
    plt.show()

visualize_svm()